# AXI tests 10 and 11: does the ORDER a graph is written in matter, and can every item be read in a block of its own? (Qwen2.5-7B-Instruct, free Colab T4)
Two tests in one notebook, both about the **tensor level** idea the owner shared on 15 Oct 2026: a model reads where things are from their *position in the text*, and (second idea) a state could be split into blocks that do not look at each other.

* **Test 10, the order test** (`docs/LAYOUT_TEST.md`). The SAME graph (70 services, 120 edges) and the SAME request ("delete the service at (r,c)", where the reply must list every edge that hangs on it) are written in five ways: sorted, shuffled, each service followed by its outgoing edges, each service followed by ALL its edges (so every edge appears twice), and turned a quarter turn. Does the model find more of the service's edges when they are written next to it? (In Run C of the earlier test it found 4 in 10 of the edges that start at a service and almost none of those that end at it.) **13 reading rules.**
* **Test 11, the block test** (`docs/KV_BLOCK_TEST.md`). Every item of a 4,000-token state is read in a forward pass of its own (the header in view, the other items not), the pieces are put side by side, and the request reads them all. If a stock model keeps its answers, deleting an item is exact and adding one costs only that item. **14 reading rules.**

**The predictions and the reading rules of both tests were written and locked BEFORE this run** in the two pages named above (each line with my probability). Nothing in this notebook changes them. A line that comes out NO is a finding, not an error. The text that gave the idea is a claim, not a result; neither test tests the remedies it proposes that need a trained model.

**What to know**
* **Cost: $0.** A local open model, no key, no paid service. It needs the **free T4 GPU**: Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save.
* **Time (my estimate, not measured):** the order test (Cell 8) 35 to 45 minutes; the block test run B1 (Cell 9) 25 to 30 minutes; run B2 (Cell 10, optional) 25 to 30 minutes. **Cell 7 measures your machine and prints the real projection before you commit.** You do not have to do everything in one sitting.
* **If Colab disconnects:** reconnect, run Cells 2 to 6 again, then run the run-cell that stopped again. Each finished problem or episode is saved to your Google Drive, so it continues where it stopped.
* **Cell 6 is a safety check on the real model** (17 lines). If it says STOP, do not run the cells after it: send the printout.
* **Do not paste any key or password anywhere.** This notebook needs none.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-tensor.zip` (this one, NOT the zip of an earlier notebook)

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and results are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_tensor")

### Cell 5: load the model (downloads about 15 GB once, unless this session already did it for an earlier notebook; about 5 minutes)

In [ ]:
import json, time, torch
from google.colab import files
from axi.experiments.kv_drop import KV_MODEL, preflight, print_verdict_kv
from axi.experiments.kv_large import large_script
from axi.experiments.kv_block import (BLOCK_SEED, BLOCK_B1_EPISODES, BLOCK_B1_STEPS, BLOCK_B2_EPISODES, BLOCK_B2_STEPS, preflight_blocks, run_block, print_block, verdict_block)
from axi.experiments.kv_layout import make_problems, kit_for, realize, run_layout, print_layout, verdict_layout, print_verdict_layout
from axi.experiments.kv_torch import build_kv, make_hf_kit

engine, tok, table, model = build_kv(KV_MODEL, True, "cuda")
kit_plain = make_hf_kit(tok, hint_rules=False)      # the system prompt of every earlier 7B test, no rule (run B1)
kit_rule = make_hf_kit(tok, hint_rules=True)        # the same prompt with the rule stated (the order test and run B2)
T0 = time.time()
def prog(i, n): print(f"  {i}/{n}   {time.time() - T0:6.0f} s", end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
res_layout = res_b1 = res_b2 = None
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1), "| model:", KV_MODEL)

### Cell 6: SAFETY CHECK on the real model (about 3 minutes). It prints one line per check and stops if any fails: the cache operations of the earlier tests (9 lines), the new 'read the blocks side by side' operation (7 lines) and the tokens of the order test's extra prompt (1 line). If it says STOP, do not run the next cells: send me the printout

In [ ]:
checks = preflight(engine, kit_plain, BLOCK_SEED, large_script) + preflight_blocks(engine, kit_plain, BLOCK_SEED, large_script)
problems = make_problems()
g_, t_, items_, _ = realize(problems[0], "adjacency")
checks.append(("the order test's adjacency prompt (one more sentence in the system prompt) is the tokenizer's own tokens", bool(kit_for(kit_rule, "adjacency").canonical(items_, t_.instruction)), "header + items + request against the whole text"))
for name, ok, detail in checks: print(("ok    " if ok else "FAIL  ") + name + "   [" + detail + "]")
json.dump([dict(name=n, ok=bool(o), detail=d) for n, o, d in checks], open("/content/out/axi_tensor_preflight.json", "w"), indent=1); store.persist("axi_tensor_preflight.json")
assert all(ok for _, ok, _ in checks), "STOP: a check failed. Do not run the next cells. Send the printout above."
print("\nALL CHECKS OK. Go on to Cell 7.")

### Cell 7: quick check of both tests (about 4 to 6 minutes): 2 problems x 5 orders, and 1 episode of 3 steps with the four ways of the block test. It tells you how long the long runs will take on this machine. If it fails, stop and send me the red error text

In [ ]:
smoke_problems = [p for p in problems if p["kind"] == "delete_dep"][:1] + [p for p in problems if p["kind"] != "delete_dep"][:1]
smoke_layout = run_layout(engine, kit_rule, table, "/content/smoke_layout.jsonl", smoke_problems, meta=dict(model=KV_MODEL, smoke=True))
print_layout(smoke_layout["summary"], "smoke, 1 delete + 1 add problem")
per_problem = sum(v["seconds"] for v in smoke_layout["summary"].values())
smoke_block = run_block(engine, kit_plain, table, 1, 3, 99, False, "/content/smoke_block.jsonl", meta=dict(model=KV_MODEL, smoke=True), script_fn=large_script)
print_block(smoke_block["summary"], "smoke, 1 episode x 3 steps")
step_s = sum(v["seconds"] for v in smoke_block["summary"].values() if v.get("n"))
print(f"\nOne problem under all five orders took {per_problem:.1f} s; one step of all four block-test ways took {step_s:.1f} s (the first steps include warm-up and the first build, so the long runs are probably quicker).")
print(f"Projection: order test (50 problems) about {50 * per_problem / 60:.0f} minutes; block test run B1 ({BLOCK_B1_EPISODES * BLOCK_B1_STEPS} steps) about {BLOCK_B1_EPISODES * BLOCK_B1_STEPS * step_s / 60:.0f} minutes; run B2 ({BLOCK_B2_EPISODES * BLOCK_B2_STEPS} steps) about {BLOCK_B2_EPISODES * BLOCK_B2_STEPS * step_s / 60:.0f} minutes.")

### Cell 8: THE ORDER TEST. 50 problems (42 'delete a service' + 8 'add'), each read under 5 orders (the counter shows problem i/50; the rule is stated in the prompt)

In [ ]:
T0 = time.time(); name = "axi_layout"
print("progress file:", store.restore(name + ".jsonl"))
res_layout = run_layout(engine, kit_rule, table, store.path(name + ".jsonl"), problems, meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog))
print(); print_layout(res_layout["summary"], "ORDER TEST: rule stated")
print_verdict_layout(verdict_layout(res_layout["summary"]), "ORDER TEST")
files.download(save(res_layout, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 9: THE BLOCK TEST, run B1: no rule in the prompt. 3 episodes x 30 steps, four ways (the counter shows episode i/3)

In [ ]:
T0 = time.time(); name = "axi_block_b1"
print("progress file:", store.restore(name + ".jsonl"))
res_b1 = run_block(engine, kit_plain, table, BLOCK_B1_EPISODES, BLOCK_B1_STEPS, BLOCK_SEED, False, store.path(name + ".jsonl"), meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog), script_fn=large_script)
print(); print_block(res_b1["summary"], "RUN B1: no rule")
print_verdict_kv(verdict_block(res_b1["summary"]), "RUN B1 (lines 12 to 14 need Run B2)")
files.download(save(res_b1, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 10 (optional): THE BLOCK TEST, run B2: the rule stated, so the reply has to find the service's edges. 3 episodes x 20 steps. Run it if you have the time; the three lines about it read 'not run' otherwise

In [ ]:
assert res_b1, "Run Cell 9 first."
T0 = time.time(); name = "axi_block_b2"
print("progress file:", store.restore(name + ".jsonl"))
res_b2 = run_block(engine, kit_rule, table, BLOCK_B2_EPISODES, BLOCK_B2_STEPS, BLOCK_SEED, True, store.path(name + ".jsonl"), meta=dict(model=KV_MODEL), progress=store.progress(name + ".jsonl", prog), script_fn=large_script)
print(); print_block(res_b2["summary"], "RUN B2: rule stated")
print_verdict_kv(verdict_block(res_b1["summary"], res_b2["summary"]), "RUN B1 and RUN B2")
files.download(save(res_b2, store.path(name + ".json"))); store.persist(name + ".json")
torch.cuda.empty_cache()

### Cell 11: the reading rules of both tests, all at once (prints nothing for a test you skipped; 'not run' for a line whose run you skipped)

In [ ]:
assert res_layout or res_b1, "Run Cell 8 or Cell 9 first."
final_layout = verdict_layout(res_layout["summary"]) if res_layout else None
final_block = verdict_block(res_b1["summary"], res_b2["summary"] if res_b2 else None) if res_b1 else None
if final_layout: print_verdict_layout(final_layout, "LAYOUT TEST, ALL LINES")
if final_block: print_verdict_kv(final_block, "BLOCK TEST, ALL LINES")
print("peak GPU memory (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 1), "| minutes since the last run started:", round((time.time() - T0) / 60, 1))

### Done
Files downloaded: `axi_layout.json` (Cell 8), `axi_block_b1.json` (Cell 9) and `axi_block_b2.json` if you ran Cell 10; the checks are also in `axi_tensor_preflight.json` on your Drive (folder `axi_tensor`). Upload the JSON files to the chat, with a screenshot of the printed tables and of the reading-rule lines. Do not paste any keys anywhere.

When the numbers are in, `docs/LAYOUT_TEST.md` and `docs/KV_BLOCK_TEST.md` each get a "Result" section **below** the locked predictions; nothing above it changes.